In [1]:
import torch
import torch.nn as nn 
import random, math
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
import matplotlib

In [2]:
IMAGE_SIZE=32
EMDED_DIM=64
ATTENTION_HEAD=4
BATCH_SIZE=12
EPOCHS=10
LR=3e-10
TEMP=0.07

In [3]:
device=torch.device('mps' if torch.mps.is_available() else 'cpu')

In [4]:

colors = ['red', 'green', 'blue', 'yellow', 'purple', 'orange', 'pink', 'brown', 'gray']
shapes = ['square', 'circle', 'triangle']
positions = ['left', 'center', 'right', 'top', 'bottom', 'top-left', 'top-right', 'bottom-left', 'bottom-right']


In [5]:

from PIL import Image, ImageDraw

IMG_SIZE = 64

def draw_sample(color, shape, position, img_size=IMG_SIZE):
    img = Image.new('RGB', (img_size, img_size), 'white')
    draw = ImageDraw.Draw(img)

    margin = 6
    h = w = img_size - 2 * margin

    # Calculate x coordinates
    if 'left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'top-left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'bottom-left' in position:
        x0 = margin
        x1 = margin + w // 2
    elif 'right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    elif 'top-right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    elif 'bottom-right' in position:
        x0 = margin + w // 2
        x1 = img_size - margin
    else:  # center or vertical positions
        x0 = margin + w // 4
        x1 = margin + 3 * w // 4

    # Calculate y coordinates
    if 'top' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'top-left' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'top-right' in position:
        y0 = margin
        y1 = margin + h // 2
    elif 'bottom' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    elif 'bottom-left' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    elif 'bottom-right' in position:
        y0 = margin + h // 2
        y1 = img_size - margin
    else:  # center or horizontal positions
        y0 = margin + h // 4
        y1 = margin + 3 * h // 4

    if shape == 'square':
        draw.rectangle([x0, y0, x1, y1],
                       fill=color, outline='black')
    elif shape == 'circle':
        draw.ellipse([x0, y0, x1, y1],
                     fill=color, outline='black')
    else:  # triangle
        draw.polygon(
            [(x0 + (x1 - x0) // 2, y0),
             (x0, y1),
             (x1, y1)],
            fill=color,
            outline='black'
        )

    return img


In [6]:
#building the dataset 
import numpy as np

class shapedataset():
    def __init__(self):
        self.images=[]
        self.captions=[]

        for c in colors:
            for s in shapes:
                for p in positions:
                    img=draw_sample(c,s,p)
                    cap=f"{c} {s} {p}"

                    self.images.append(torch.from_numpy(np.array(img, copy=True)).permute(2, 0, 1).float() / 255.0)
                    self.captions.append(cap)

        self.vocab, self.word2idx=self.build_vocab(self.captions)

    def build_vocab(self,texts):
        words= sorted({w for t in texts for w in t.split()})
        vocab= ['[CLS]']+ words
        w2i={w:i for i,w in enumerate(vocab)}
        return vocab,w2i
    def encode_text(self,text):
        toks = [self.word2idx['[CLS]']] + [self.word2idx[w] for w in text.split()]
        return torch.tensor(toks, dtype=torch.long)
    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
         return self.images[i], self.encode_text(self.captions[i]), self.captions[i]


full_ds = shapedataset()
VOCAB_SIZE = len(full_ds.vocab)

In [7]:
print(VOCAB_SIZE)
print(full_ds.vocab)

22
['[CLS]', 'blue', 'bottom', 'bottom-left', 'bottom-right', 'brown', 'center', 'circle', 'gray', 'green', 'left', 'orange', 'pink', 'purple', 'red', 'right', 'square', 'top', 'top-left', 'top-right', 'triangle', 'yellow']


In [9]:
# Train-val dataset

train_size = int(0.8 * len(full_ds))
val_size = len(full_ds) - train_size

train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [train_size, val_size]
)

In [10]:
train_loader= DataLoader(train_ds, batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

In [11]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.convolutions=nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32,
                      kernel_size=3, stride=2, padding=1 ),
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64,
                              kernel_size=3, stride=2, padding=1 ),
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128,
                                  kernel_size=3, stride=2, padding=1 ), 
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=256,
                                  kernel_size=3, stride=2, padding=1 ) ,
            nn.ReLU(), 

        )

        self.projection=nn.Linear(256, 64)
        self.layernorm1=nn.LayerNorm(64)
    def forward(self,x):
        x= self.convolutions(x)
        x= x.mean(dim=[2,3]) #along height, width
        x=self.projection(x)
        x=F.normalize(self.layernorm1(x), dim=-1)
        return x



In [12]:
class TextEncoder(nn.Module):
    def __init__(self, embed_dim=EMDED_DIM, num_heads=ATTENTION_HEAD, vocab_size=VOCAB_SIZE, context_size=4):
        super().__init__()

        self.token_embedding=nn.Embedding(vocab_size, embed_dim)
        self.postion_embedding=nn.Embedding(vocab_size, context_size)
        self.mha= nn.MultiheadAttention(embed_dim, num_heads,)
        self.norm=nn.LayerNorm(embed_dim)

    def forward(self,x):
        pass

In [26]:
import torch
import torch.nn as nn

class ImageFeatureMap(nn.Module):
    def __init__(self):
        super().__init__()

        self.Conv = nn.Sequential(
            # Standard Conv: 224 -> 112
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Depthwise Conv: 112 -> 112
            nn.Conv2d(32, 32, 3, stride=1, padding=1,
                      groups=32, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            # Pointwise Conv: 32 -> 64 channels
            nn.Conv2d(32, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Depthwise Conv: 112 -> 56
            nn.Conv2d(64, 64, 3, stride=2, padding=1,
                      groups=64, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            # Pointwise Conv: 64 -> 128 channels
            nn.Conv2d(64, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            # Depthwise Conv: 56 -> 56
            nn.Conv2d(128, 128, 3, stride=1, padding=1,
                      groups=128, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True), 

            #PointWise Conv: 128 -> 128 channels 
            nn.Conv2d(in_channels=128, out_channels=128,
                      stride=1, groups=1, bias=False, padding=0, kernel_size=(1,1)), 
            nn.BatchNorm2d(128), 
            nn.ReLU(inplace=True), 


            #DepthWise Conv 128 -> 128 channels

            nn.Conv2d(in_channels=128, out_channels=128, groups=128, 
                      stride=2, padding=1, bias=False, kernel_size=(3,3)), 
            nn.BatchNorm2d(128), 
            nn.ReLU(inplace=True), 


            #Pointwise Conv 128 -> 256 channels, 56x56 -> 28x28 
            nn.Conv2d(
                   in_channels=128,
                   out_channels=256,
                   groups=1,
                   bias=False,
                   stride=1,
                   kernel_size=(1, 1)
                     ),
                   nn.BatchNorm2d(256),
                   nn.ReLU(inplace=True)
                   )

    def forward(self, x):
        x=self.Conv(x)

        return x


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        embedding_dimension=256,
        num_heads=4,
        dropout=0.1,
        qkv_bias=False
    ):
        super().__init__()

        assert embedding_dimension % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = embedding_dimension // num_heads
        self.embedding_dimension = embedding_dimension

        self.w_k = nn.Linear(embedding_dimension, embedding_dimension, bias=qkv_bias)
        self.w_q = nn.Linear(embedding_dimension, embedding_dimension, bias=qkv_bias)
        self.w_v = nn.Linear(embedding_dimension, embedding_dimension, bias=qkv_bias)

        self.attn_dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(embedding_dimension, embedding_dimension)
        self.proj_dropout = nn.Dropout(dropout)

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.w_k(x)
        queries = self.w_q(x)
        values = self.w_v(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attention_score = queries @ keys.transpose(-2, -1)

        attention_weights = torch.softmax(
            attention_score / (self.head_dim ** 0.5),
            dim=-1
        )

        attention_weights = self.attn_dropout(attention_weights)

        context_vector = attention_weights @ values

        context_vector = context_vector.transpose(1, 2).contiguous()
        context_vector = context_vector.view(
            b, num_tokens, self.embedding_dimension
        )

        context_vector = self.out_proj(context_vector)
        context_vector = self.proj_dropout(context_vector)

        return context_vector


class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=256,out_channels=256, kernal_size=4, stride=4, padding=0, bias=False):
        super().__init__()

        self.conv=nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=(kernal_size,kernal_size),
                  stride=stride, padding=padding , bias=bias)

    def forward(self,x ):
        x=self.conv(x)
        x= torch.flatten(x, start_dim=2, end_dim=-1)
        x= x.transpose(1,2) #[batch, token, embedding_dimension]
        return x
    


In [27]:
image= torch.randn(3,224,224)
images=torch.stack((image,image), dim=0)
images.shape

torch.Size([2, 3, 224, 224])

In [28]:
def test(x):
    mha=MultiHeadAttention()
    feature_map=ImageFeatureMap()
    patch_embedding=PatchEmbedding()

    return mha(patch_embedding(feature_map(x)))


    

In [29]:
context_vectors=test(images)

In [30]:
context_vectors.shape

torch.Size([2, 49, 256])